
# Project 1 — Discrete-Time Asset Allocation

This notebook implements:

- 1 benchmark
- 3 formal experiments
- 1 robustness check (risk-aversion sensitivity)|


In [2]:

from __future__ import annotations
from dataclasses import dataclass
import numpy as np
import pandas as pd
import time
from IPython.display import display



## 1. Core configuration
Notes:

- `n` counts only risky assets, so the portfolio weight vector has length `n+1` including cash.
- `alpha` is the CARA risk-aversion coefficient.


In [3]:

@dataclass
class Config:
    T: int
    n: int
    r: float
    mu: np.ndarray
    var: np.ndarray
    alpha: float
    init_wealth: float = 1.0
    init_weights: np.ndarray | None = None
    turnover_cap: float = 0.10
    weight_step: float = 0.10
    wealth_min: float = 0.40
    wealth_max: float = 2.50
    wealth_points: int = 81
    mc_samples: int = 300
    return_floor: float = -0.95
    seed: int = 0

    def __post_init__(self) -> None:
        assert self.T < 10
        assert self.n < 5
        assert len(self.mu) == self.n
        assert len(self.var) == self.n
        if self.init_weights is None:
            self.init_weights = np.ones(self.n + 1) / (self.n + 1)
        assert len(self.init_weights) == self.n + 1
        assert np.all(self.init_weights >= 0.0)
        assert abs(self.init_weights.sum() - 1.0) < 1e-10


## 2. Grid helpers

In [4]:

def compositions(total: int, parts: int):
    if parts == 1:
        yield (total,)
        return
    for x in range(total + 1):
        for tail in compositions(total - x, parts - 1):
            yield (x,) + tail


def generate_weight_grid(n_risky: int, step: float) -> np.ndarray:
    units = int(round(1.0 / step))
    grid = np.array(list(compositions(units, n_risky + 1)), dtype=float) / units
    return grid


def turnover(w_old: np.ndarray, w_new: np.ndarray) -> float:
    return 0.5 * np.abs(w_old - w_new).sum()


def precompute_feasible_actions(weight_grid: np.ndarray, cap: float):
    feasible = []
    for w in weight_grid:
        l1 = np.abs(weight_grid - w[None, :]).sum(axis=1)
        feasible_idx = np.where(l1 <= 2.0 * cap + 1e-12)[0]
        feasible.append(feasible_idx)
    return feasible


def sample_risky_returns(cfg: Config, rng: np.random.Generator, m: int) -> np.ndarray:
    R = rng.normal(loc=cfg.mu, scale=np.sqrt(cfg.var), size=(m, cfg.n))
    R = np.maximum(R, cfg.return_floor)
    return R


## 3. Transition and nearest-grid utilities

In [5]:

def transition(
    wealth: float,
    target_w: np.ndarray,
    risky_returns: np.ndarray,
    cash_r: float
) -> tuple[np.ndarray, np.ndarray]:
    m = risky_returns.shape[0]
    gross = np.empty((m, target_w.size))
    gross[:, 0] = 1.0 + cash_r
    gross[:, 1:] = 1.0 + risky_returns

    portfolio_gross = gross @ target_w
    wealth_next = wealth * portfolio_gross
    weights_next = (gross * target_w[None, :]) / portfolio_gross[:, None]
    return wealth_next, weights_next


def nearest_weight_index(w: np.ndarray, weight_grid: np.ndarray) -> int:
    return int(np.argmin(np.abs(weight_grid - w[None, :]).sum(axis=1)))


def nearest_wealth_index(x: float, wealth_grid: np.ndarray) -> int:
    return int(np.argmin(np.abs(wealth_grid - x)))


def batch_nearest_weight_indices(weights: np.ndarray, weight_grid: np.ndarray) -> np.ndarray:
    # weights: (m, d), grid: (G, d)
    # returns nearest grid index for each row in weights
    dists = np.abs(weights[:, None, :] - weight_grid[None, :, :]).sum(axis=2)
    return np.argmin(dists, axis=1)


def interp_values_batch(
    V_next: np.ndarray,          # shape (num_weight_states, num_wealth_points)
    wealth_grid: np.ndarray,
    next_weight_idx: np.ndarray, # shape (m,)
    wealth_next: np.ndarray      # shape (m,)
) -> np.ndarray:
    out = np.empty(len(wealth_next))
    clipped = np.clip(wealth_next, wealth_grid[0], wealth_grid[-1])
    for i in range(len(wealth_next)):
        out[i] = np.interp(clipped[i], wealth_grid, V_next[next_weight_idx[i]])
    return out



## 4. Optimized backward induction


In [6]:

def precompute_action_dynamics(cfg: Config, weight_grid: np.ndarray, common_returns: list[np.ndarray]):
    # For each t and action a_idx, precompute:
    # 1) portfolio gross factors under the sampled returns
    # 2) next weights
    # 3) nearest next weight-grid indices
    dyn = []
    num_actions = len(weight_grid)

    for t in range(cfg.T):
        R_samples = common_returns[t]
        m = R_samples.shape[0]

        gross = np.empty((m, cfg.n + 1))
        gross[:, 0] = 1.0 + cfg.r
        gross[:, 1:] = 1.0 + R_samples

        dyn_t = []
        for a_idx in range(num_actions):
            target_w = weight_grid[a_idx]
            portfolio_gross = gross @ target_w                          # shape (m,)
            next_weights = (gross * target_w[None, :]) / portfolio_gross[:, None]
            next_weight_idx = batch_nearest_weight_indices(next_weights, weight_grid)
            dyn_t.append({
                "portfolio_gross": portfolio_gross,
                "next_weight_idx": next_weight_idx,
            })
        dyn.append(dyn_t)
    return dyn


def solve_backward_induction(cfg: Config):
    rng = np.random.default_rng(cfg.seed)

    weight_grid = generate_weight_grid(cfg.n, cfg.weight_step)
    feasible_actions = precompute_feasible_actions(weight_grid, cfg.turnover_cap)
    wealth_grid = np.linspace(cfg.wealth_min, cfg.wealth_max, cfg.wealth_points)

    num_w = len(weight_grid)
    num_x = len(wealth_grid)

    V = [np.zeros((num_w, num_x)) for _ in range(cfg.T + 1)]
    Pi = [np.zeros((num_w, num_x), dtype=int) for _ in range(cfg.T)]

    terminal = -np.exp(-cfg.alpha * wealth_grid)
    V[cfg.T][:] = terminal[None, :]

    common_returns = [sample_risky_returns(cfg, rng, cfg.mc_samples) for _ in range(cfg.T)]
    dyn = precompute_action_dynamics(cfg, weight_grid, common_returns)

    for t in reversed(range(cfg.T)):
        for w_idx, w in enumerate(weight_grid):
            action_list = feasible_actions[w_idx]
            for x_idx, wealth in enumerate(wealth_grid):
                best_q = -np.inf
                best_action = int(action_list[0])

                for a_idx in action_list:
                    info = dyn[t][a_idx]
                    wealth_next = wealth * info["portfolio_gross"]
                    next_values = interp_values_batch(
                        V_next=V[t + 1],
                        wealth_grid=wealth_grid,
                        next_weight_idx=info["next_weight_idx"],
                        wealth_next=wealth_next
                    )
                    q_val = next_values.mean()

                    if q_val > best_q:
                        best_q = q_val
                        best_action = a_idx

                V[t][w_idx, x_idx] = best_q
                Pi[t][w_idx, x_idx] = best_action

    return weight_grid, wealth_grid, V, Pi



## 5. Vectorized policy simulation



In [7]:

def simulate_policy(cfg: Config, weight_grid, wealth_grid, Pi, n_paths: int = 5000):
    rng = np.random.default_rng(cfg.seed + 123)

    wealth = np.full(n_paths, cfg.init_wealth, dtype=float)
    weights = np.repeat(cfg.init_weights[None, :], n_paths, axis=0)
    avg_turnover_by_t = []

    for t in range(cfg.T):
        risky_R = sample_risky_returns(cfg, rng, n_paths)

        # Find current state indices
        w_idx = batch_nearest_weight_indices(weights, weight_grid)
        wealth_clipped = np.clip(wealth, wealth_grid[0], wealth_grid[-1])
        x_idx = np.abs(wealth_clipped[:, None] - wealth_grid[None, :]).argmin(axis=1)

        # Choose action for each path
        a_idx = Pi[t][w_idx, x_idx]
        target_w = weight_grid[a_idx]  # shape (n_paths, n+1)

        # Turnover
        turnovers = 0.5 * np.abs(weights - target_w).sum(axis=1)

        # Transition for all paths in one shot
        gross = np.empty((n_paths, cfg.n + 1))
        gross[:, 0] = 1.0 + cfg.r
        gross[:, 1:] = 1.0 + risky_R

        portfolio_gross = np.sum(gross * target_w, axis=1)
        wealth = wealth * portfolio_gross
        weights = (gross * target_w) / portfolio_gross[:, None]

        avg_turnover_by_t.append(float(turnovers.mean()))

    expected_utility = np.mean((1.0 - np.exp(-cfg.alpha * wealth)) / cfg.alpha)
    certainty_equivalent = -np.log(np.mean(np.exp(-cfg.alpha * wealth))) / cfg.alpha

    return {
        "mean_terminal_wealth": float(np.mean(wealth)),
        "std_terminal_wealth": float(np.std(wealth)),
        "expected_utility": float(expected_utility),
        "certainty_equivalent": float(certainty_equivalent),
        "avg_turnover_by_t": avg_turnover_by_t,
    }


## 6. Experiment runner

In [8]:

def summarize_run(cfg: Config, label: str, n_paths: int = 3000) -> dict:
    start = time.time()
    weight_grid, wealth_grid, V, Pi = solve_backward_induction(cfg)
    result = simulate_policy(cfg, weight_grid, wealth_grid, Pi, n_paths=n_paths)
    elapsed = time.time() - start

    init_w_idx = nearest_weight_index(cfg.init_weights, weight_grid)
    init_x_idx = nearest_wealth_index(cfg.init_wealth, wealth_grid)
    first_action_idx = Pi[0][init_w_idx, init_x_idx]
    first_action = weight_grid[first_action_idx]

    row = {
        "label": label,
        "T": cfg.T,
        "n": cfg.n,
        "r": cfg.r,
        "alpha": cfg.alpha,
        "mu": np.round(cfg.mu, 4).tolist(),
        "var": np.round(cfg.var, 5).tolist(),
        "init_weights": np.round(cfg.init_weights, 4).tolist(),
        "mean_terminal_wealth": result["mean_terminal_wealth"],
        "std_terminal_wealth": result["std_terminal_wealth"],
        "expected_utility": result["expected_utility"],
        "certainty_equivalent": result["certainty_equivalent"],
        "avg_turnover": float(np.mean(result["avg_turnover_by_t"])),
        "avg_turnover_by_t": np.round(result["avg_turnover_by_t"], 4).tolist(),
        "runtime_sec": elapsed,
        "t0_optimal_target": np.round(first_action, 4).tolist(),
    }
    return row


## 7. Benchmark + 3 formal experiments + 1 robustness check

In [9]:

def run_benchmark() -> pd.DataFrame:
    cfg = Config(
        T=5,
        n=1,
        r=0.002,
        mu=np.array([0.010]),
        var=np.array([0.0009]),
        alpha=3.0,
        init_weights=np.array([0.5, 0.5]),
        turnover_cap=1.0,
        weight_step=0.05,
        wealth_min=0.5,
        wealth_max=1.8,
        wealth_points=81,
        mc_samples=300,
        seed=1,
    )
    return pd.DataFrame([summarize_run(cfg, label="benchmark_single_asset")])


def run_experiment_1_baseline() -> pd.DataFrame:
    cfg = Config(
        T=5,
        n=3,
        r=0.002,
        mu=np.array([0.006, 0.010, 0.014]),
        var=np.array([0.0004, 0.0009, 0.0016]),
        alpha=3.0,
        init_weights=np.array([0.40, 0.20, 0.20, 0.20]),
        turnover_cap=0.10,
        weight_step=0.10,
        wealth_min=0.5,
        wealth_max=1.8,
        wealth_points=61,
        mc_samples=200,
        seed=7,
    )
    return pd.DataFrame([summarize_run(cfg, label="exp1_baseline")])


def run_experiment_2_horizon_sweep() -> pd.DataFrame:
    rows = []
    for T in [2, 3, 5, 7, 9]:
        cfg = Config(
            T=T,
            n=3,
            r=0.002,
            mu=np.array([0.006, 0.010, 0.014]),
            var=np.array([0.0004, 0.0009, 0.0016]),
            alpha=3.0,
            init_weights=np.array([0.40, 0.20, 0.20, 0.20]),
            turnover_cap=0.10,
            weight_step=0.10,
            wealth_min=0.5,
            wealth_max=2.2,
            wealth_points=61,
            mc_samples=180,
            seed=10 + T,
        )
        rows.append(summarize_run(cfg, label=f"exp2_horizon_T{T}"))
    return pd.DataFrame(rows)


def run_experiment_3_dimension_sweep() -> pd.DataFrame:
    rows = []

    cfg_n3 = Config(
        T=5,
        n=3,
        r=0.002,
        mu=np.array([0.006, 0.010, 0.014]),
        var=np.array([0.0004, 0.0009, 0.0016]),
        alpha=3.0,
        init_weights=np.array([0.40, 0.20, 0.20, 0.20]),
        turnover_cap=0.10,
        weight_step=0.10,
        wealth_min=0.5,
        wealth_max=1.8,
        wealth_points=61,
        mc_samples=200,
        seed=21,
    )
    rows.append(summarize_run(cfg_n3, label="exp3_dimension_n3"))

    cfg_n4 = Config(
        T=5,
        n=4,
        r=0.002,
        mu=np.array([0.005, 0.008, 0.010, 0.013]),
        var=np.array([0.0003, 0.0006, 0.0012, 0.0020]),
        alpha=3.0,
        init_weights=np.array([0.30, 0.20, 0.20, 0.15, 0.15]),
        turnover_cap=0.10,
        weight_step=0.10,
        wealth_min=0.5,
        wealth_max=1.8,
        wealth_points=61,
        mc_samples=200,
        seed=22,
    )
    rows.append(summarize_run(cfg_n4, label="exp3_dimension_n4"))

    return pd.DataFrame(rows)


def run_experiment_4_risk_aversion_sensitivity() -> pd.DataFrame:
    rows = []
    for alpha in [1.0, 3.0, 5.0]:
        cfg = Config(
            T=5,
            n=3,
            r=0.002,
            mu=np.array([0.006, 0.010, 0.014]),
            var=np.array([0.0004, 0.0009, 0.0016]),
            alpha=alpha,
            init_weights=np.array([0.40, 0.20, 0.20, 0.20]),
            turnover_cap=0.10,
            weight_step=0.10,
            wealth_min=0.5,
            wealth_max=1.8,
            wealth_points=61,
            mc_samples=200,
            seed=30 + int(alpha * 10),
        )
        rows.append(summarize_run(cfg, label=f"robust_alpha_{alpha}"))
    return pd.DataFrame(rows)


def run_all_experiments():
    results = {
        "Benchmark": run_benchmark(),
        "Experiment 1: Baseline": run_experiment_1_baseline(),
        "Experiment 2: Horizon Sweep": run_experiment_2_horizon_sweep(),
        "Experiment 3: Dimension Sweep": run_experiment_3_dimension_sweep(),
        "Experiment 4: Risk-Aversion Sensitivity": run_experiment_4_risk_aversion_sensitivity(),
    }
    return results


## 8. Run experiments and display tables

In [10]:

results = run_all_experiments()

for name, df in results.items():
    print(name)
    display(df)


Benchmark


,label,T,n,r,alpha,mu,var,init_weights,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,avg_turnover_by_t,runtime_sec,t0_optimal_target
0,benchmark_single_asset,5,1,0.002,3.0,[0.01],[0.0009],"[0.5, 0.5]",1.052778,0.06907,0.318864,1.045694,0.1,"[0.5, 0.0, 0.0, 0.0, 0.0]",115.953395,"[0.0, 1.0]"


Experiment 1: Baseline


,label,T,n,r,alpha,mu,var,init_weights,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,avg_turnover_by_t,runtime_sec,t0_optimal_target
0,exp1_baseline,5,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.051929,0.051469,0.318962,1.047976,0.10192,"[0.1, 0.1022, 0.1027, 0.1033, 0.1015]",389.643537,"[0.3, 0.2, 0.2, 0.3]"


Experiment 2: Horizon Sweep


,label,T,n,r,alpha,mu,var,init_weights,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,avg_turnover_by_t,runtime_sec,t0_optimal_target
0,exp2_horizon_T2,2,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.015918,0.019321,0.317485,1.015359,0.101592,"[0.1, 0.1032]",142.125254,"[0.3, 0.2, 0.3, 0.2]"
1,exp2_horizon_T3,3,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.028415,0.030818,0.318028,1.026989,0.101446,"[0.1, 0.1019, 0.1024]",242.497310,"[0.3, 0.2, 0.2, 0.3]"
2,exp2_horizon_T5,5,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.054651,0.048262,0.319099,1.051155,0.102389,"[0.1, 0.102, 0.1027, 0.1033, 0.1039]",353.482905,"[0.3, 0.2, 0.2, 0.3]"
3,exp2_horizon_T7,7,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.080809,0.073973,0.319989,1.072674,0.101636,"[0.1, 0.1021, 0.1025, 0.1033, 0.1015, 0.1023, ...",550.336872,"[0.3, 0.2, 0.2, 0.3]"
4,exp2_horizon_T9,9,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.106000,0.078935,0.320919,1.096751,0.099313,"[0.1, 0.102, 0.1026, 0.1034, 0.1017, 0.1037, 0...",629.446080,"[0.3, 0.2, 0.2, 0.3]"


Experiment 3: Dimension Sweep


,label,T,n,r,alpha,mu,var,init_weights,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,avg_turnover_by_t,runtime_sec,t0_optimal_target
0,exp3_dimension_n3,5,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.051918,0.049973,0.318972,1.048196,0.10197,"[0.1, 0.1022, 0.1026, 0.1035, 0.1016]",540.375787,"[0.3, 0.2, 0.2, 0.3]"
1,exp3_dimension_n4,5,4,0.002,3.0,"[0.005, 0.008, 0.01, 0.013]","[0.0003, 0.0006, 0.0012, 0.002]","[0.3, 0.2, 0.2, 0.15, 0.15]",1.048669,0.049091,0.318837,1.045084,0.11431,"[0.15, 0.1035, 0.1043, 0.1069, 0.1069]",2669.670116,"[0.2, 0.2, 0.2, 0.1, 0.3]"


Experiment 4: Risk-Aversion Sensitivity


,label,T,n,r,alpha,mu,var,init_weights,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,avg_turnover_by_t,runtime_sec,t0_optimal_target
0,robust_alpha_1.0,5,3,0.002,1.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.051738,0.050377,0.650227,1.050472,0.102011,"[0.1, 0.1021, 0.1027, 0.1035, 0.1017]",792.891624,"[0.3, 0.2, 0.2, 0.3]"
1,robust_alpha_3.0,5,3,0.002,3.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.051006,0.050609,0.318928,1.047179,0.102037,"[0.1, 0.1022, 0.1027, 0.1034, 0.1019]",377.806008,"[0.3, 0.2, 0.2, 0.3]"
2,robust_alpha_5.0,5,3,0.002,5.0,"[0.006, 0.01, 0.014]","[0.0004, 0.0009, 0.0016]","[0.4, 0.2, 0.2, 0.2]",1.051992,0.051373,0.198926,1.045456,0.102040,"[0.1, 0.1022, 0.1028, 0.1034, 0.1018]",380.677269,"[0.3, 0.2, 0.2, 0.3]"



## 9. Compact summary table




In [11]:

summary_table = pd.concat(
    [df.assign(experiment=name) for name, df in results.items()],
    ignore_index=True
)

cols = [
    "experiment", "label", "T", "n", "alpha",
    "mean_terminal_wealth", "std_terminal_wealth",
    "expected_utility", "certainty_equivalent",
    "avg_turnover", "runtime_sec", "t0_optimal_target"
]

display(summary_table[cols])


,experiment,label,T,n,alpha,mean_terminal_wealth,std_terminal_wealth,expected_utility,certainty_equivalent,avg_turnover,runtime_sec,t0_optimal_target
0,Benchmark,benchmark_single_asset,5,1,3.0,1.052778,0.069070,0.318864,1.045694,0.100000,115.953395,"[0.0, 1.0]"
1,Experiment 1: Baseline,exp1_baseline,5,3,3.0,1.051929,0.051469,0.318962,1.047976,0.101920,389.643537,"[0.3, 0.2, 0.2, 0.3]"
2,Experiment 2: Horizon Sweep,exp2_horizon_T2,2,3,3.0,1.015918,0.019321,0.317485,1.015359,0.101592,142.125254,"[0.3, 0.2, 0.3, 0.2]"
3,Experiment 2: Horizon Sweep,exp2_horizon_T3,3,3,3.0,1.028415,0.030818,0.318028,1.026989,0.101446,242.497310,"[0.3, 0.2, 0.2, 0.3]"
4,Experiment 2: Horizon Sweep,exp2_horizon_T5,5,3,3.0,1.054651,0.048262,0.319099,1.051155,0.102389,353.482905,"[0.3, 0.2, 0.2, 0.3]"
5,Experiment 2: Horizon Sweep,exp2_horizon_T7,7,3,3.0,1.080809,0.073973,0.319989,1.072674,0.101636,550.336872,"[0.3, 0.2, 0.2, 0.3]"
6,Experiment 2: Horizon Sweep,exp2_horizon_T9,9,3,3.0,1.106000,0.078935,0.320919,1.096751,0.099313,629.446080,"[0.3, 0.2, 0.2, 0.3]"
7,Experiment 3: Dimension Sweep,exp3_dimension_n3,5,3,3.0,1.051918,0.049973,0.318972,1.048196,0.101970,540.375787,"[0.3, 0.2, 0.2, 0.3]"
8,Experiment 3: Dimension Sweep,exp3_dimension_n4,5,4,3.0,1.048669,0.049091,0.318837,1.045084,0.114310,2669.670116,"[0.2, 0.2, 0.2, 0.1, 0.3]"
9,Experiment 4: Risk-Aversion Sensitivity,robust_alpha_1.0,5,3,1.0,1.051738,0.050377,0.650227,1.050472,0.102011,792.891624,"[0.3, 0.2, 0.2, 0.3]"
